# 🌍 OpenAQ Dataset — Daily Update

**Purpose:** Fetch fresh air quality data from the OpenAQ v3 API, aggregate to daily and monthly country-level averages, and publish as a Kaggle dataset.

**Schedule:** Run daily via Kaggle Scheduled Notebooks.

**Output files** (written to `/kaggle/working/`):
- `openaq_locations.csv` — all active monitoring stations worldwide
- `openaq_daily.csv` — daily PM2.5 / PM10 averages per station
- `openaq_monthly.csv` — monthly country-level averages (ready for WHO/World Bank join)
- `update_log.csv` — run history: date, rows fetched, countries covered

**API:** [OpenAQ v3](https://api.openaq.org) — requires `OPENAQ_API_KEY` Kaggle Secret.

## 1. Install dependencies

In [ ]:
%%capture
!pip install tenacity loguru

# Clone the project repo so src/ is importable
import os
from pathlib import Path
REPO_DIR = Path('/kaggle/working/repo')
if not REPO_DIR.exists():
    os.system('git clone --depth 1 https://github.com/olegsher-ds-2025/naya_prokect.git /kaggle/working/repo')
print('repo ready:', REPO_DIR.exists())

## 2. Setup

In [ ]:
import os
import sys
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
from loguru import logger

# ── Paths ──────────────────────────────────────────────────────────────────────
ON_KAGGLE = Path('/kaggle').exists()
OUTPUT    = Path('/kaggle/working') if ON_KAGGLE else Path('data/output')
REPO_ROOT = Path('/kaggle/working/repo') if ON_KAGGLE else Path('.')
OUTPUT.mkdir(parents=True, exist_ok=True)

# Add repo root to path so 'from src.xxx import ...' works
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── API key ────────────────────────────────────────────────────────────────────
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret('OPENAQ_API_KEY')
else:
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.environ['OPENAQ_API_KEY']

assert API_KEY, 'OPENAQ_API_KEY not found — add it as a Kaggle Secret'

# ── FULL_LOAD toggle ───────────────────────────────────────────────────────
# Set True ONCE for initial historical backfill, then revert to False
FULL_LOAD = False
DATE_TO   = date.today()
DATE_FROM = date(2023, 1, 1) if FULL_LOAD else DATE_TO - timedelta(days=8)

print(f'Mode     : {"FULL LOAD" if FULL_LOAD else "incremental"}')
print(f'Fetching : {DATE_FROM} → {DATE_TO}')
print(f'Output   : {OUTPUT}')

## 3. Fetch locations

In [ ]:
from src.openaq_client import OpenAQClient
from src.pipeline import build_locations_df, PRIORITY_CITIES

# 0.5 req/s stays within OpenAQ free-tier rate limits (~10-60 req/min)
client = OpenAQClient(API_KEY, requests_per_second=0.5)

# Fetch only the 13 priority cities (top-10 mortality + LA/Fresno/Houston)
# instead of a global fetch — reduces sensors from ~1,300 to ~250
locations_df = build_locations_df(client, cities=PRIORITY_CITIES)


## 4. Fetch measurements & aggregate to daily

In [ ]:
from src.pipeline import build_daily_df

daily_df = build_daily_df(
    client,
    locations_df,
    date_from=DATE_FROM,
    date_to=DATE_TO,
    max_workers=8,
)

print(f'\nDaily records fetched : {len(daily_df):,}')
print(f'Date range            : {daily_df["date"].min()} → {daily_df["date"].max()}')
print(f'Locations covered     : {daily_df["location_id"].nunique():,}')
print(f'Countries covered     : {daily_df["country_code"].nunique()}')

daily_df.head(5)

## 5. Aggregate to monthly country level

In [ ]:
from src.pipeline import build_monthly_country_df

monthly_df = build_monthly_country_df(daily_df)

# Quick look at top polluted countries (latest month, PM2.5)
latest_month = monthly_df['year_month'].max()
top20 = (
    monthly_df[
        (monthly_df['year_month'] == latest_month) &
        (monthly_df['parameter'] == 'pm25')
    ]
    .nlargest(20, 'value_mean_ugm3')
    [['country_code','country_name','value_mean_ugm3','station_count','exceeds_who_guideline']]
)

print(f'\nMonthly records : {len(monthly_df):,}')
print(f'Latest month    : {latest_month}')
print(f'\nTop 20 most polluted countries (PM2.5, {latest_month}):')
top20

## 6. Validation

In [ ]:
import matplotlib.pyplot as plt

# Quick bar chart — top 20 countries by PM2.5
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(
    top20['country_name'][::-1],
    top20['value_mean_ugm3'][::-1],
    color=['#d73027' if v else '#4575b4' for v in top20['exceeds_who_guideline'][::-1]]
)
ax.axvline(15, color='red', linestyle='--', linewidth=1, label='WHO guideline (15 µg/m³)')
ax.set_xlabel('PM2.5 µg/m³')
ax.set_title(f'Top 20 Countries by PM2.5 — {latest_month}')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'top20_pm25_preview.png', dpi=150)
plt.show()

# Data quality checks
assert daily_df['value_ugm3'].ge(0).all(), 'Negative PM values found!'
assert daily_df['country_code'].notna().all(), 'Null country codes found!'
assert daily_df['date'].str.match(r'\d{4}-\d{2}-\d{2}').all(), 'Bad date format!'
print('\n✅ All validation checks passed')

## 7. Save outputs

In [ ]:
# If incremental, merge with existing data to avoid losing history
def merge_with_existing(new_df: pd.DataFrame, path: Path, key_cols: list) -> pd.DataFrame:
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_df], ignore_index=True)
        return combined.drop_duplicates(subset=key_cols, keep='last')
    return new_df

# Locations (always overwrite — reflects current station state)
locations_path = OUTPUT / 'openaq_locations.csv'
locations_df.to_csv(locations_path, index=False)

# Daily (merge with existing to preserve history)
daily_path = OUTPUT / 'openaq_daily.csv'
daily_full = merge_with_existing(
    daily_df, daily_path,
    key_cols=['date', 'location_id', 'parameter']
)
daily_full.to_csv(daily_path, index=False)

# Monthly (merge and recalculate)
monthly_path = OUTPUT / 'openaq_monthly.csv'
monthly_full = build_monthly_country_df(daily_full)
monthly_full.to_csv(monthly_path, index=False)

# Update log
log_path = OUTPUT / 'update_log.csv'
log_row = pd.DataFrame([{
    'run_date':          str(date.today()),
    'fetch_from':        str(DATE_FROM),
    'fetch_to':          str(DATE_TO),
    'new_daily_records': len(daily_df),
    'total_daily_records': len(daily_full),
    'countries':         daily_df['country_code'].nunique(),
    'stations':          daily_df['location_id'].nunique(),
}])
if log_path.exists():
    log_row = pd.concat([pd.read_csv(log_path), log_row], ignore_index=True)
log_row.to_csv(log_path, index=False)

print('Files written to', OUTPUT)
for f in sorted(OUTPUT.glob('*.csv')):
    rows = sum(1 for _ in open(f)) - 1
    print(f'  {f.name:<35} {rows:>8,} rows')

print(f'\n✅ Dataset update complete — {date.today()}')